**<font color=skyblue>Encoding dataset by facenet_pytorch</font>**

本程式配合 facenetPytorch_build_dataset.py 建立人像資料集後，對個別資料集進行人臉擷取與編碼（以 pickle file 儲存），最後對新的人像照片進行辨識。
- 第一部分：對個別資料集進行人臉擷取與編碼 The first part is to encode the dataset stored in facenet_data/..., and stores the encoddings in a pickle file for later use.
- 第二部分： 對新的人像照片進行人像擷取並與現存的編碼進行比對辨識 The second part is to read in an image, encode it and then compare the similarity with the previous encoddings by Cosine similarity and Euclidean distance.

<font color=yellow>Import Packages</font>

In [ ]:
import os
import torch
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

<font color=yellow>第一部分：Encodding a dataset stored in a directory (e.g. facenet_dataset) with several subdirectories</font>

In [ ]:
# Initialize MTCNN and InceptionResnetV1
device = 'cpu'
mtcnn = MTCNN(image_size=160, select_largest=False, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Directory containing subdirectories with images
dir_path = 'facenet_dataset'

# List to hold face encodings and labels
face_encodings = [] # List to hold face encodings for each detected face. Each encoding is a 512-dimensional vector.
labels = [] # List to hold labels corresponding to each encoding. Use subdirectory names as labels.

# Iterate over subdirectories in directory
for subdir in os.listdir(dir_path):
    subdir_path = os.path.join(dir_path, subdir)
    
    # Skip if not a directory
    if not os.path.isdir(subdir_path):
        continue

    # Iterate over images in subdirectory
    for filename in os.listdir(subdir_path):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            # Load image
            img_path = os.path.join(subdir_path, filename)
            img = Image.open(img_path)

            # Detect face
            face = mtcnn(img)

            # Skip if no face is detected
            if face is None:
                continue

            # Move face tensor to cuda
            face = face.to(device)

            # Encode face using the InceptionResnetV1 model
            encoding = resnet(face.unsqueeze(0)).detach().cpu()

            # Append encoding and label to lists
            face_encodings.append(encoding)
            labels.append(subdir)  # Use subdirectory name as label

# Save encodings and labels to a pickle file
with open('encodings.pkl', 'wb') as f:
    pickle.dump((face_encodings, labels), f)
print("Encodings and labels saved to encodings.pkl")

<font color=yellow>第二部分：Apply the encoddings to detect the faces of an image</font>

- Read in an image with multiple faces, compare the faces with the encodings in the pickle file, and print the labels:

- This script uses the cosine similarity to compare the face encodings with the dataset encodings. It assumes that the face encodings and the dataset encodings are L2-normalized, which is the case if they are produced by the InceptionResnetV1 model. The label of the most similar face encoding in the dataset is printed for each face.

<font color=yellow>將照片上的所有人臉與已經編碼儲存的資料檔（encodings.pkl）一一比對，並列出最可能的人名（sub-directory 名稱）

In [ ]:
device = 'cpu'
# Load encodings and labels from pickle file
with open('encodings.pkl', 'rb') as f:
    dataset_encodings, dataset_labels = pickle.load(f)

# Convert list of tensors to a 2D tensor
dataset_encodings = torch.cat(dataset_encodings)

# Load image with multiple faces
img_path = 'pictures/facenet_test.jpg'
img = Image.open(img_path)

# Detect faces in the image
mtcnn = MTCNN(keep_all=True)
faces = mtcnn(img, return_prob=False)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# List to hold face encodings
face_encodings = []

# Encode each face using the InceptionResnetV1 model
for face in faces:
    face = face.to(device)
    encoding = resnet(face.unsqueeze(0)).detach().cpu()
    face_encodings.append(encoding)

# Convert list of tensors to a 2D tensor
face_encodings = torch.cat(face_encodings)

# Compute cosine similarity between each face encoding and the dataset encodings
# the threshold is 0.5?
similarity = cosine_similarity(face_encodings, dataset_encodings) 

# Get the index of the most similar face encoding in the dataset for each face
most_similar_indices = np.argmax(similarity, axis=1)

# Print the label of the most similar face encoding in the dataset for each face
print('Most similar faces in cosine similarity:')
for i in most_similar_indices:
    print(dataset_labels[i])

# Compute Euclidean distance between each face encoding and the dataset encodings
# the threshold is 0.5?
distances = euclidean_distances(face_encodings, dataset_encodings)

# Get the index of the most similar (smallest distance) face encoding in the dataset for each face
most_similar_indices = np.argmin(distances, axis=1)

# Print the label of the most similar face encoding in the dataset for each face
print('Most similar faces by Euclidean distance:')
for i in most_similar_indices:
    print(dataset_labels[i])

<font color=yellow>同上，設定相似度的標準，並在照片上印出人名。若不符合標準，則打 'X'</font>

In [ ]:
# Display the image
fig, ax = plt.subplots()
ax.imshow(img)

# Get the bounding boxes of the faces
boxes, _ = mtcnn.detect(img)

# Compute Euclidean distance between each face encoding and the dataset encodings
distances = euclidean_distances(face_encodings, dataset_encodings)

# Get the index of the most similar (smallest distance) face encoding in the dataset for each face
most_similar_indices = np.argmin(distances, axis=1)
# closet_distance = np.min(distances, axis=1)

# Annotate each face with the most similar label

for cnt, (box, i) in enumerate(zip(boxes, most_similar_indices)):
    # set the threshold at 0.6
    if distances[cnt, i] < 0.7:
        label = dataset_labels[i]
    else:
        label = 'X'  # Label as 'X' if distance is above threshold
   
    # label = dataset_labels[i]
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], linewidth=1, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    plt.text(box[0], box[1], label, color='red')

plt.axis('off')
plt.show()